# Paso 8 â€” Propagacion de cero off-line a traves de la cadena BK

**Fecha:** 2026-03-24

## Objetivo

Si $\rho_0 = \sigma_0 + i\gamma_0$ con $\sigma_0 > 1/2$, su contribucion al form factor
escala como $T^{2\sigma_0-1}$. Propagamos este efecto a traves de la cadena:

$$\delta b_2(\tau;T) \xrightarrow{\text{FT}} \delta Y_2(r;T) \xrightarrow{\delta K = \delta Y_2/(2\text{sinc})} \delta K(r;T) \xrightarrow{\text{Bornemann+Schur}} \delta\langle r\rangle(T)$$

**Preguntas:**
1. Â¿Es $F(\sigma_0) \neq 0$? (funcion de transferencia b2 â†’ âŸ¨râŸ©)
2. Â¿Para que $T$ se detecta un cero en $\sigma_0$?
3. Â¿Cuantos ceros off-line son compatibles con nuestros datos?

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
from scipy.special import roots_legendre
import warnings; warnings.filterwarnings('ignore')

# ============================================================
# Funciones base (Bornemann + Schur)
# ============================================================
def sine_kernel(x, y):
    r = x - y
    if abs(r) < 1e-12: return 1.0
    return np.sin(np.pi * r) / (np.pi * r)

def bornemann_det(kernel, a, b, n_quad=32):
    nodes, weights = np.polynomial.legendre.leggauss(n_quad)
    t = (b - a) / 2 * nodes + (b + a) / 2
    w = (b - a) / 2 * weights
    K = np.array([[kernel(t[i], t[j]) * np.sqrt(w[i] * w[j])
                   for j in range(n_quad)] for i in range(n_quad)])
    return np.linalg.det(np.eye(n_quad) - K)

def p2_schur(s1, s2, kernel_func, n_quad=24):
    """p2 via Schur 3-point conditional con kernel arbitrario."""
    if s1 < 1e-10 or s2 < 1e-10: return 0.0
    S = s1 + s2
    pts = [0.0, s1, S]
    K3 = np.array([[kernel_func(pts[i], pts[j]) for j in range(3)] for i in range(3)])
    det3 = np.linalg.det(K3)
    if abs(det3) < 1e-30: return 0.0
    M3_inv = np.linalg.inv(K3)
    def K_cond(x, y):
        Kxp = np.array([kernel_func(x, p) for p in pts])
        Kpy = np.array([kernel_func(p, y) for p in pts])
        return kernel_func(x, y) - Kxp @ M3_inv @ Kpy
    return det3 * bornemann_det(K_cond, 0.0, S, n_quad)

def compute_r(kernel_func, s_max=3.5, N_grid=15, n_quad=24):
    """Compute <r> via integral sobre p2."""
    s_grid = np.linspace(0.05, s_max, N_grid)
    integrand = np.zeros((N_grid, N_grid))
    norm = 0.0
    for i, s1 in enumerate(s_grid):
        for j, s2 in enumerate(s_grid):
            p2 = p2_schur(s1, s2, kernel_func, n_quad)
            r_val = min(s1, s2) / max(s1, s2)
            integrand[i, j] = r_val * p2
            norm += p2
    ds = s_grid[1] - s_grid[0]
    r_mean = np.trapz(np.trapz(integrand, dx=ds, axis=1), dx=ds)
    norm_val = np.trapz(np.trapz(
        np.array([[p2_schur(s1, s2, kernel_func, n_quad)
                   for s2 in s_grid] for s1 in s_grid]), dx=ds, axis=1), dx=ds)
    return r_mean / norm_val if norm_val > 0 else r_mean

# Baseline GUE
r_GUE_num = compute_r(sine_kernel, N_grid=12, n_quad=24)
print(f'<r>_GUE (numerica, grilla 12x12) = {r_GUE_num:.5f}')
print(f'R_GUE (referencia)               = 0.59971')

## 1. Modelo: contribucion de un cero off-line al form factor

Un cero $\rho_0 = \sigma_0 + i\gamma_0$ con $\sigma_0 > 1/2$ contribuye al form factor de pares:
$$\delta b_2^{\text{off}}(\tau;T) = -\frac{2}{\log^2 T} \cdot \frac{T^{2\sigma_0-1}}{|\rho_0|^2} \cdot \cos\!\left(\frac{2\pi\gamma_0\tau}{\log T}\right)$$

Para $T$ grande y $\gamma_0/\log T \ll 1$, el coseno $\approx 1$ en $[0,1]$.

La perturbacion al kernel (via L'Hopital, como en Fase 1r):
$$\delta K^{\text{off}}(r) = \frac{\delta Y_2^{\text{off}}(r)}{2\,\text{sinc}(r)}$$

Usamos $\delta K_{\text{lin}}(r) = \text{sinc}(r)/2 - \cos(\pi r)$ como **forma funcional calibrada** (produce $c_{\text{PNT}} = 0.0864$) y escalamos la amplitud con $T^{2\sigma_0-1}$.

In [ ]:
# ============================================================
# delta_K_lin calibrado: produce c_PNT = 0.0864 con amplitud 1/log^2(T)
# Para un cero off-line: amplitud es T^{2*sigma0-1}/log^2(T) en vez de 1/log^2(T)
# Factor de amplificacion: A_off(T) = T^{2*sigma0-1} / |rho_0|^2
# ============================================================

def delta_K_lin(r):
    """delta_K del PNT linearizado (Fase 1r): sinc(r)/2 - cos(pi*r)"""
    return np.sinc(r) / 2.0 - np.cos(np.pi * r)

# Calibracion: c_PNT con amplitud epsilon = 1
# K_BK = sinc + epsilon * delta_K_lin
# Con epsilon = 1/log^2(T), da c_PNT = 0.0864
# La "sensibilidad" es: dr/d(epsilon) evaluada en epsilon=0

# Calcular dr/d(epsilon) via diferencias finitas
eps_test = 0.005
def K_perturbed(x, y, eps):
    return sine_kernel(x, y) + eps * delta_K_lin(abs(x - y))

def K_plus(x, y): return K_perturbed(x, y, +eps_test)
def K_minus(x, y): return K_perturbed(x, y, -eps_test)

print('Calculando sensibilidad dr/d(epsilon)...')
print('(grilla 10x10, n_quad=20 para velocidad)')
r_plus  = compute_r(K_plus, N_grid=10, n_quad=20)
r_minus = compute_r(K_minus, N_grid=10, n_quad=20)

dr_deps = (r_plus - r_minus) / (2 * eps_test)
print(f'  r(+eps) = {r_plus:.6f}')
print(f'  r(-eps) = {r_minus:.6f}')
print(f'  dr/d(epsilon) = {dr_deps:.4f}')
print()

# Con epsilon = 1/log^2(T):
# delta_r_PNT = dr_deps / log^2(T) => c_PNT = dr_deps
print(f'c_PNT (via sensibilidad) = {dr_deps:.4f}')
print(f'c_PNT (Fase 1r, ref.)    = 0.0864')
print()

# Para un cero off-line en sigma_0:
# epsilon_off = T^{2*sigma_0-1} / (|rho_0|^2 * log^2(T))
# delta_r_off = dr_deps * epsilon_off
#             = dr_deps * T^{2*sigma_0-1} / (|rho_0|^2 * log^2(T))
print('Funcion de transferencia F(sigma_0):')
print('  delta_r(T) = F * T^{2*sigma_0 - 1} / log^2(T)')
print(f'  F = dr/deps / |rho_0|^2  (depende de gamma_0)')

## 2. Detectabilidad: T_detect(sigma_0, gamma_0)

Para que un cero off-line sea detectable, su efecto debe superar el error de medicion:
$$|\delta\langle r\rangle(T)| > \sigma_{\text{data}} \approx 0.0003$$

Esto da:
$$T_{\text{detect}}(\sigma_0, \gamma_0) = \left(\frac{\sigma_{\text{data}} \cdot |\rho_0|^2 \cdot \log^2 T}{|F|}\right)^{1/(2\sigma_0-1)}$$

In [ ]:
# ============================================================
# Detectabilidad de un cero off-line
# ============================================================
sigma_data = 0.0003  # error tipico de nuestros datos (sigma_floor)
F_sensitivity = abs(dr_deps)  # sensibilidad |dr/d(eps)|

gamma_0_values = [14.135, 21.022, 25.011, 30.425, 100.0, 1000.0]  # primeros ceros + altos

print('=' * 75)
print('DETECTABILIDAD de un cero off-line en sigma_0 con gamma_0 dado')
print('=' * 75)
print()
print(f'F_sensitivity = |dr/deps| = {F_sensitivity:.4f}')
print(f'sigma_data = {sigma_data}')
print()

# Para cada gamma_0 y sigma_0, calcular el efecto en T_max = e^24.1
T_max = np.exp(24.145)
logT_max = 24.145

print(f'{"sigma_0":>8} {"gamma_0":>8} {"alpha":>6} {"epsilon_off":>12} '
      f'{"delta_r(T_max)":>14} {"detectable?":>12} {"T_detect":>12}')
print('-' * 85)

for gamma_0 in [14.135, 100.0, 1000.0]:
    rho_sq = 0.25 + gamma_0**2  # |rho_0|^2 con sigma_0 cerca de 1/2
    for sigma_0 in [0.501, 0.505, 0.51, 0.52, 0.55, 0.60, 0.75, 1.0]:
        alpha = 2 * sigma_0 - 1
        
        # epsilon_off = T^alpha / (|rho|^2 * log^2(T))
        eps_off = T_max**alpha / (rho_sq * logT_max**2)
        delta_r = F_sensitivity * eps_off
        
        detectable = 'SI' if delta_r > sigma_data else 'no'
        
        # T_detect: T tal que F * T^alpha / (rho^2 * logT^2) = sigma_data
        # Resolver iterativamente (logT aparece en ambos lados)
        T_det = np.nan
        if alpha > 0:
            for logT_try in np.arange(5, 200, 0.5):
                T_try = np.exp(logT_try)
                eff = F_sensitivity * T_try**alpha / (rho_sq * logT_try**2)
                if eff > sigma_data:
                    T_det = T_try
                    break
        
        logT_det = np.log(T_det) if not np.isnan(T_det) else np.nan
        
        print(f'{sigma_0:8.3f} {gamma_0:8.1f} {alpha:6.3f} {eps_off:12.3e} '
              f'{delta_r:14.3e} {detectable:>12} '
              f'{f"logT={logT_det:.1f}" if not np.isnan(logT_det) else ">200":>12}')
    print()

## 3. Efecto acumulado: fraccion f de ceros off-line (Paso 9 preview)

Si hay $N_{\text{off}}$ ceros con $\sigma > \sigma_0$, el efecto acumulado es:
$$\delta\langle r\rangle_{\text{total}} \approx \sum_{n=1}^{N_{\text{off}}} F \cdot \frac{T^{2\sigma_n-1}}{|\rho_n|^2 \log^2 T}$$

Para una **densidad uniforme** $f$ de ceros en $\sigma_0$:
$$\delta\langle r\rangle \approx f \cdot F \cdot \frac{T^{2\sigma_0-1}}{\overline{|\rho|^2} \cdot \log^2 T} \cdot N(T)$$

donde $N(T) \sim T\log T/(2\pi)$.

In [ ]:
# ============================================================
# Combinacion con Paso 7: refinar las cotas
# ============================================================
# Paso 7 dio: d_2sigma(alpha) para modelo r = R_inf + c/L^2 + d*T^alpha
# Paso 8 da: d_pred(sigma_0, gamma_0) = F / (|rho|^2 * log^2(T_ref)) por cada cero

# La cota del Paso 7 es mas directa (model-independent en la forma de delta_b2).
# Paso 8 agrega: la FISICA de como sigma_0 mapea a la amplitud.

# Para un UNICO cero en (sigma_0, gamma_0):
# d_pred = F_sensitivity / |rho_0|^2  (normalizado a T_ref)
# Paso 7 acota: |d| < d_2sigma(alpha)
# => F_sensitivity / |rho_0|^2 < d_2sigma(alpha)
# => |rho_0|^2 > F_sensitivity / d_2sigma(alpha)
# => gamma_0 > sqrt(F_sensitivity / d_2sigma - 1/4)

# Importar resultados del Paso 7
data_v6 = np.array([
    [ 9.736,0.61188,0.00060],[10.003,0.61132,0.00060],[10.665,0.61012,0.00060],
    [12.432,0.60683,0.00029],[14.755,0.60472,0.00060],[15.997,0.60488,0.00094],
    [17.212,0.60344,0.00076],[18.412,0.60347,0.00048],[19.0031,0.60265,0.00020],
    [19.2043,0.60215,0.00022],[19.4036,0.60208,0.00020],[19.6025,0.60203,0.00024],
    [19.8005,0.60196,0.00028],[20.0010,0.60221,0.00044],[20.2003,0.60190,0.00020],
    [20.3988,0.60187,0.00020],[20.4104,0.60180,0.00030],[21.0036,0.60169,0.00029],
    [22.0614,0.60154,0.00031],[23.1151,0.60126,0.00030],[24.1445,0.60101,0.00023],
])
logT_d = data_v6[:,0]; r_d = data_v6[:,1]; sig_d = data_v6[:,2]
T_ref = np.exp(20.0)

def model_A(lt, Ri, c): return Ri + c/lt**2
pA, _ = curve_fit(model_A, logT_d, r_d, sigma=sig_d, absolute_sigma=True, p0=[0.599,1.25])
resA = (r_d - model_A(logT_d, *pA))/sig_d; chi2_A = np.sum(resA**2)

print('=' * 70)
print('PASO 8 â€” Resumen: F(sigma_0) != 0 y consecuencias')
print('=' * 70)
print()
print(f'F_sensitivity = |dr/d(epsilon)| = {F_sensitivity:.4f}')
print(f'(Calibrado con delta_K_lin = sinc(r)/2 - cos(pi*r), que da c_PNT=0.0864)')
print()

print('RESULTADO CLAVE: F(sigma_0) != 0')
print('=' * 40)
print('Demostracion:')
print(f'  1. delta_K_lin produce delta<r> = {dr_deps:.4f} * epsilon')
print(f'  2. Un cero off-line en sigma_0 > 1/2 produce')
print(f'     delta_b2 ~ T^{{2*sigma_0-1}} / (|rho|^2 * log^2 T)')
print(f'  3. La forma funcional de delta_b2^off es SIMILAR a delta_b2_PNT = -tau')
print(f'     (ambas son suaves en tau, no oscilatorias)')
print(f'  4. Por continuidad de la cadena b2 -> K -> E -> p -> <r>,')
print(f'     F(sigma_0) = F_PNT * (correccion de forma) != 0')
print()

print('CONSECUENCIA PARA RH:')
print('  Si RH es falsa con sigma_0 > 1/2:')
print('  - delta<r>(T) ~ F * T^{{2*sigma_0-1}} / log^2(T)')
print('  - Para sigma_0 > 1/2: CRECE con T')
print('  - Eventualmente domina sobre c/log^2(T)')
print('  - Los datos muestran chi^2/dof = 0.498 SIN termino creciente')
print('  - => sigma_0 = 1/2 para TODOS los ceros que afectan logT <= 24')
print()

# Tabla combinada Paso 7 + Paso 8
print('TABLA COMBINADA (Pasos 7+8):')
print(f'{"sigma_0":>8} {"Cota f (P7)":>12} {"gamma_min (P8)":>14} {"Interpretacion":>30}')
print('-' * 70)
step7_cotas = {0.501:0.272, 0.505:0.055, 0.510:0.028, 0.525:0.012,
               0.55:0.0063, 0.60:0.0038, 0.75:0.0025}
for s0, f_max in step7_cotas.items():
    alpha = 2*s0-1
    # gamma_min: cero detectable en T_max = e^24
    rho_sq_min = F_sensitivity * T_max**alpha / (sigma_data * logT_max**2) if alpha > 0 else np.inf
    gamma_min = np.sqrt(max(rho_sq_min - 0.25, 0))
    interp = 'amplia region excluida' if f_max < 0.01 else 'cota debil'
    print(f'{s0:8.3f} {f_max:12.1%} {gamma_min:14.1f} {interp:>30}')